# Cavity_Example — 腔与 TWS: 1D 腔表/TWS/3D 场图/TDS

本 notebook 是官方算例系列示例之一 (全部位于 examples/ 目录): 只跑这一个官方算例, 逐步解读输入卡、展示本算例最有代表性的图。

In [ ]:
%run ../notebooks/_bootstrap.py

In [ ]:
# ===== 共享规格 (单一数据源, 与 06 汇总一致) =====
from examples._examples_spec import (
    EXAMPLES, run_example, phase_files, compare_xemit)
NAME = "Cavity_Example"
STEM = "astra"
print("算例:", NAME, "| 物理:", EXAMPLES[NAME]["title"])


In [ ]:
# ===== 运行本算例 (generator/astra 按需) =====
work = run_example(NAME)
print("输出文件:")
for f in sorted(work.glob(STEM + ".*")):
    print("  ", f.name)

In [ ]:
# ===== 束团统计 (最后一个 z 位置) =====
from astra_tools.io import read_distribution
from astra_tools.analysis.statistics import compute_statistics, print_statistics
from astra_tools.widgets.panels import stats_table_html
ph = phase_files(work, STEM)
dist = read_distribution(ph[-1])
print("相空间文件:", ph[-1].name)
print_statistics(compute_statistics(dist),
                 title="%s @ %s" % (NAME, ph[-1].name))
stats_table_html(compute_statistics(dist))

### 示例要点

本例是场图大户: 1D 腔表 (3_cell_L-Band)、TWS 行波结构、3D 场图 (3D_test)。它的 golden PScan 也展示单腔余弦律。

In [ ]:
import numpy as _np
import matplotlib.pyplot as plt
from astra_tools.io.field_map import (read_cavity_field, parse_field_map_file,
                                       expand_tws_field_map)
from astra_tools.plot.field_plots import plot_cavity_field
cav = read_cavity_field(work / "3_cell_L-Band.dat")
plot_cavity_field(cav, omega=2 * _np.pi * 1.3e9, title="3_cell_L-Band")
# TWS 行波结构: 4 值头 (z1 z2 n m) 周期展开 (deck 里 C_numb=9)
attrs, data = parse_field_map_file(work / "TWS_Sband.dat")
zf, ff = expand_tws_field_map(data[:, 0], data[:, 1], attrs["z1"], attrs["z2"],
                              m_cells_in_body=attrs["m"], n_cell=9)
plt.figure()
plt.plot(zf * 1e3, ff, lw=0.8, label="TWS 9 cells (normalized)")
plt.xlabel("z [mm]")
plt.ylabel("Ez [a.u.] (× MaxE=40 MV/m)")
plt.title("TWS_Sband expanded")
plt.legend()
plt.tight_layout()


In [ ]:
from astra_tools.plot.advanced_plots import plot_3d_map_slices, plot_laser_on_axis
plot_3d_map_slices(work / "3D_test.ex", axis="z", n_slices=3, unit="V/m")
plot_laser_on_axis(work / "3D_test.ex", unit="V/m")

In [ ]:
from astra_tools.io.astra_misc import read_pscan
from astra_tools.plot.advanced_plots import plot_phase_scan, plot_pscan_compression
pscan = read_pscan(PROJECT_ROOT / "examples/Cavity_Example/golden/astra.PScan.001")
plot_phase_scan(pscan)   # E(phi) = E0 + A cos(phi-phi0)
plot_pscan_compression(pscan)

In [ ]:
# ===== 黄金比对 (rel < 0.5% 判 OK) =====
compare_xemit(NAME, work)
print("其他官方算例的示例 notebook 同样位于 examples/ 目录")